In [21]:
import os

date = "250804"
rnn_folder = f"D:/Python_TK_3/datas/{date}_DL"

shap_number = "ssStim_evoked_03"
#os.mkdir(f"{rnn_folder}/model_{shap_number}")

In [22]:
fs = 20  # サンプリング周波数
calc_start = 5
calc_end = 39
experiments = 59

# データのサンプリングレート
first_ex = 0
#last_ex = pupil1.shape[0] + pupil2.shape[0] + pupil3.shape[0]
last_ex = 58
NumberOfDatas = last_ex - first_ex + 1        # number of experiments
start_stim = 20
stop_stim = 20.5
start_ave= 10
end_ave= 20
look_frame = 10               # read the previous and next n frames as input

In [23]:
import numpy as np

peak_idxs = np.load(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_peak_frame.npy")

print(peak_idxs.shape)

(59,)


In [8]:
import numpy as np
import tqdm

all_evo = []
for ex in tqdm.tqdm(range(experiments)):
    if ex < 10:
        img_path = os.path.join("D:/Python_TK_3/datas/250623_GCaMP/250331-250421_HindPaw_trace2/imgs2", f"250623_ex0{ex}.npy")
    if ex > 9:
        img_path = os.path.join("D:/Python_TK_3/datas/250623_GCaMP/250331-250421_HindPaw_trace2/imgs2", f"250623_ex{ex}.npy")
    mov = np.load(img_path)

    evoked_mov = mov[int(fs*14)-1:int(fs*19)+1]
    all_evo.append(evoked_mov)

print(len(all_evo), all_evo[0].shape)

  0%|          | 0/59 [00:00<?, ?it/s]

100%|██████████| 59/59 [00:18<00:00,  3.17it/s]

59 (102, 128, 135)


In [9]:
import numpy as np

raw_pupil  = np.load(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_pre_pupil.npy")
print(raw_pupil.shape)

(59, 24)


In [10]:
pupil = raw_pupil

datasets_x = []
datasets_y = []
for ex in tqdm.tqdm(range(experiments)):
    dataset_x = []
    dataset_y = []
    for f in range(pupil.shape[1]):
        dataset_y.append(pupil[ex, f])
        predata = []
        for step in range(look_frame+1):
            img_3ch = np.stack((all_evo[ex][f+step], all_evo[ex][f+1+step], all_evo[ex][f+2+step]), axis=0)
            predata.append(img_3ch)
        dataset_x.append(predata)
    datasets_x.append(dataset_x)
    datasets_y.append(dataset_y)

datasets_x = np.array(datasets_x)
datasets_y = np.array(datasets_y)

print(datasets_x.shape, datasets_y.shape)

100%|██████████| 59/59 [00:01<00:00, 32.94it/s]


(59, 24, 11, 3, 128, 135) (59, 24)


In [11]:
selected_ex = np.load(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_selected_experiments_excluded.npy")
print(selected_ex)

[ 0  1  2  3  5  6  7  8  9 12 13 15 16 18 19 23 24 25 28 29 31 32 34 35
 36 41 42 44 45 46 47 48 49 50 52 53 54 56 57 58]


In [12]:
dataset_x = datasets_x[selected_ex]
dataset_y = datasets_y[selected_ex]

print(dataset_x.shape, dataset_y.shape)

(40, 24, 11, 3, 128, 135) (40, 24)


In [13]:
# Data allocation from experiment No.
TRAIN = list(range(24))
VALID = list(range(8))
TEST  = list(range(8))

In [14]:
import random

# 1. 0～59 のソート済みリストを用意
numbers = list(range(len(dataset_x)))
# （任意）結果を再現可能にする場合はシードを固定
random.seed(123)

# 2. 最初に 40 個をランダム抽出
train_numbers = random.sample(numbers, len(TRAIN))

# 3. 残りの要素を求める
remaining = list(set(numbers) - set(train_numbers))

# 4. 残りからさらに 10 個をランダム抽出
valid_numbers = random.sample(remaining, len(VALID))
# 5. 最後のリストは、残った要素すべて
test_numbers = list(set(remaining) - set(valid_numbers))


# 各リストの長さを確認
print(len(train_numbers), len(valid_numbers), len(test_numbers))  # ⇒ 40, 10, 10

print(train_numbers)
print(valid_numbers)
print(test_numbers)

np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_train_numbers.npy", np.array(train_numbers))
np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_valid_numbers.npy", np.array(valid_numbers))
np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_test_numbers.npy", np.array(test_numbers))

24 8 8
[3, 17, 5, 26, 38, 6, 2, 24, 21, 10, 27, 1, 37, 4, 30, 35, 25, 22, 7, 29, 0, 13, 33, 12]
[36, 32, 9, 19, 14, 34, 28, 18]
[39, 8, 11, 15, 16, 20, 23, 31]


In [15]:
pre_trainX = dataset_x[train_numbers]
trainY = dataset_y[train_numbers]
pre_validX = dataset_x[valid_numbers]
validY = dataset_y[valid_numbers]
pre_testX  = dataset_x[test_numbers]
testY = dataset_y[test_numbers]

print(pre_trainX.shape, pre_validX.shape, pre_testX.shape)

(24, 24, 11, 3, 128, 135) (8, 24, 11, 3, 128, 135) (8, 24, 11, 3, 128, 135)


In [16]:
trainX = pre_trainX.transpose(0,1,2,4,5,3)
print(trainX.shape)

validX = pre_validX.transpose(0,1,2,4,5,3)
print(validX.shape)

testX = pre_testX.transpose(0,1,2,4,5,3)
print(testX.shape)

(24, 24, 11, 128, 135, 3)
(8, 24, 11, 128, 135, 3)
(8, 24, 11, 128, 135, 3)


In [17]:
def reshape_data(x_data, y_data):
    x_data = x_data.reshape(x_data.shape[0]*x_data.shape[1], x_data.shape[2], x_data.shape[3], x_data.shape[4], x_data.shape[5])
    y_data = y_data.reshape(y_data.shape[0]*y_data.shape[1])

    return x_data, y_data

In [18]:
trainX, trainY = reshape_data(trainX, trainY)
validX, validY = reshape_data(validX, validY)
testX, testY   = reshape_data(testX, testY)

print(trainX.shape, trainY.shape)
print(validX.shape, validY.shape)
print(testX.shape, testY.shape)

(576, 11, 128, 135, 3) (576,)
(192, 11, 128, 135, 3) (192,)
(192, 11, 128, 135, 3) (192,)


In [19]:
#np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_train_signal_min.npy", signal_min)
#np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_train_signal_max.npy", signal_max)
#np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_train_signal_ave.npy", signal_ave)
#np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_train_signal_std.npy", signal_std)

In [20]:
np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_train_features.npy", trainX)
np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_train_targets.npy", trainY)
np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_valid_features.npy", validX)
np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_valid_targets.npy", validY)
np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_test_features.npy", testX)
np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_test_targets.npy", testY)